In [10]:
import numpy as np
import lightgbm as lgb
import optuna
import random
import math
from sklearn.model_selection import cross_val_score, KFold
from sklearn.datasets import load_breast_cancer
from collections import defaultdict
import warnings

In [11]:
# Tắt các cảnh báo từ Optuna để output gọn gàng hơn
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning)

In [12]:
# ----------------------------------------------------------------------------
# PHẦN 1: ĐỊNH NGHĨA BÀI TOÁN TỐI ƯU
# ----------------------------------------------------------------------------

# Tải dữ liệu và định nghĩa không gian siêu tham số
X, y = load_breast_cancer(return_X_y=True)

# Không gian tìm kiếm siêu tham số cho LightGBM
SEARCH_SPACE = {
    'learning_rate': ('float', 0.01, 0.3),
    'num_leaves': ('int', 20, 150),
    'max_depth': ('int', 3, 10),
    'subsample': ('float', 0.6, 1.0),
    'colsample_bytree': ('float', 0.6, 1.0),
}

def objective_function(params):
    """
    Hàm mục tiêu để đánh giá một bộ siêu tham số.
    Sử dụng cross-validation để có ước tính ổn định.
    Đây là "inner CV" trong mô hình Nested Cross-Validation.
    """
    model = lgb.LGBMClassifier(objective='binary', **params, random_state=42, verbosity=-1)
    cv = KFold(n_splits=3, shuffle=True, random_state=42)
    score = cross_val_score(model, X, y, cv=cv, scoring='accuracy').mean()
    return score

In [13]:
# ----------------------------------------------------------------------------
# PHẦN 2: CÁC THÀNH PHẦN CỦA AMSCO
# ----------------------------------------------------------------------------

class KnowledgeHub:
    """
    Lưu trữ và chia sẻ kiến thức (kết quả các lần thử) giữa các agent.
    """
    def __init__(self):
        # Lưu trữ các trial dưới dạng list các dict: {'agent_id', 'params', 'score'}
        self.trials = []
        self.best_score = -1.0
        self.best_params = None

    def store(self, agent_id, params, score):
        self.trials.append({'agent_id': agent_id, 'params': params, 'score': score})
        if score > self.best_score:
            self.best_score = score
            self.best_params = params
            print(f"  [KnowledgeHub] New best score: {self.best_score:.4f} from {agent_id}")

    def get_all_trials(self):
        return self.trials

    def get_best_trial(self):
        return {'params': self.best_params, 'score': self.best_score}

class StrategyAgent:
    """Lớp cơ sở cho các agent chiến lược."""
    def __init__(self, agent_id, search_space, knowledge_hub):
        self.agent_id = agent_id
        self.search_space = search_space
        self.knowledge_hub = knowledge_hub

    def run(self, budget):
        raise NotImplementedError

class RandomAgent(StrategyAgent):
    """Agent thực hiện tìm kiếm ngẫu nhiên."""
    def run(self, budget):
        print(f"    -> Running RandomAgent with budget: {budget}")
        for _ in range(budget):
            params = {}
            for name, (type, low, high) in self.search_space.items():
                if type == 'float':
                    params[name] = random.uniform(low, high)
                elif type == 'int':
                    params[name] = random.randint(low, high)
            
            score = objective_function(params)
            self.knowledge_hub.store(self.agent_id, params, score)

class BayesianAgent(StrategyAgent):
    """Agent thực hiện tối ưu hóa Bayes (sử dụng Optuna TPE)."""
    def run(self, budget):
        print(f"    -> Running BayesianAgent with budget: {budget}")
        
        # Hàm mục tiêu cho Optuna
        def optuna_objective(trial):
            params = {
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3),
                'num_leaves': trial.suggest_int('num_leaves', 20, 150),
                'max_depth': trial.suggest_int('max_depth', 3, 10),
                'subsample': trial.suggest_float('subsample', 0.6, 1.0),
                'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            }
            score = objective_function(params)
            # Lưu kết quả vào KnowledgeHub ngay sau khi có
            self.knowledge_hub.store(self.agent_id, params, score)
            return score

        study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
        
        # --- Warm-start: Chia sẻ kiến thức ---
        # Lấy các kết quả đã có từ KnowledgeHub để "mớm" cho Optuna
        existing_trials = self.knowledge_hub.get_all_trials()
        if existing_trials:
            print(f"   ... BayesianAgent warm-starting with {len(existing_trials)} previous trials.")
            for t in existing_trials:
                # Chuyển đổi params về định dạng Optuna cần
                dist = {
                    'learning_rate': optuna.distributions.FloatDistribution(0.01, 0.3),
                    'num_leaves': optuna.distributions.IntDistribution(20, 150),
                    'max_depth': optuna.distributions.IntDistribution(3, 10),
                    'subsample': optuna.distributions.FloatDistribution(0.6, 1.0),
                    'colsample_bytree': optuna.distributions.FloatDistribution(0.6, 1.0),
                }
                # Chỉ thêm những trial có tham số nằm trong không gian tìm kiếm
                try:
                    frozen_trial = optuna.trial.create_trial(
                        params=t['params'],
                        distributions=dist,
                        value=t['score']
                    )
                    study.add_trial(frozen_trial)
                except Exception:
                    # Bỏ qua nếu tham số không hợp lệ (ví dụ từ GridAgent)
                    pass

        study.optimize(optuna_objective, n_trials=budget)

class GridAgent(StrategyAgent):
    """Agent thực hiện tìm kiếm lưới cục bộ (dùng để tinh chỉnh)."""
    def run(self, budget):
        print(f"    -> Running GridAgent with budget: {budget}")
        best_params = self.knowledge_hub.get_best_trial()['params']
        if not best_params:
            print("   ... GridAgent skipped (no best params yet).")
            return

        # Tạo một lưới nhỏ xung quanh điểm tốt nhất hiện tại
        lr = best_params.get('learning_rate', 0.1)
        nl = best_params.get('num_leaves', 50)
        
        local_grid = [
            {**best_params, 'learning_rate': lr * 0.9, 'num_leaves': nl - 5},
            {**best_params, 'learning_rate': lr, 'num_leaves': nl},
            {**best_params, 'learning_rate': lr * 1.1, 'num_leaves': nl + 5},
        ]
        
        # Giới hạn số lần chạy theo budget
        for i in range(min(budget, len(local_grid))):
            params = local_grid[i]
            # Đảm bảo các giá trị vẫn nằm trong khoảng hợp lệ
            params['num_leaves'] = max(20, min(150, params['num_leaves']))
            params['learning_rate'] = max(0.01, min(0.3, params['learning_rate']))
            
            score = objective_function(params)
            self.knowledge_hub.store(self.agent_id, params, score)

class PerformanceMonitor:
    """Theo dõi hiệu suất của các agent."""
    def __init__(self, agent_ids):
        self.agent_ids = agent_ids
        self.history = defaultdict(list)

    def update(self, all_trials):
        # Đơn giản là lưu lại tất cả điểm số cho mỗi agent
        self.history = defaultdict(list)
        for trial in all_trials:
            self.history[trial['agent_id']].append(trial['score'])

    def get_agent_rewards(self):
        rewards = {}
        for agent_id in self.agent_ids:
            # Trả về danh sách rỗng nếu agent chưa có bản ghi nào
            scores = self.history.get(agent_id, [])
            if len(scores) < 2:
                # Nếu chưa có đủ dữ liệu, reward mặc định là 0.5
                rewards[agent_id] = 0.5 
            else:
                # Reward được tính bằng mức cải thiện trung bình trong 5 lần thử gần nhất
                recent_scores = scores[-5:]
                reward = np.mean(np.diff(recent_scores)) if len(recent_scores) > 1 else 0
                # Chuẩn hóa reward về khoảng  (ước lượng)
                normalized_reward = (math.tanh(reward * 100) + 1) / 2
                rewards[agent_id] = normalized_reward
        return rewards

class MetaController_UCB1:
    """
    Bộ điều khiển Meta sử dụng thuật toán UCB1 để phân bổ ngân sách.
    """
    def __init__(self, agent_ids):
        self.agent_ids = agent_ids
        self.agent_pulls = {agent_id: 0 for agent_id in agent_ids}
        self.agent_rewards = {agent_id: 0.0 for agent_id in agent_ids}
        self.total_pulls = 0

    def allocate(self, slice_budget):
        # Khởi tạo: mỗi agent chạy ít nhất 1 lần
        uninitialized_agents = [aid for aid, pulls in self.agent_pulls.items() if pulls == 0]
        if uninitialized_agents:
            # Chọn 1 agent chưa được khởi tạo
            agent_to_run = uninitialized_agents[0]
            allocations = {agent_id: 0 for agent_id in self.agent_ids}
            allocations[agent_to_run] = slice_budget
            print(f"  [MetaController] Initializing {agent_to_run}")
            return allocations

        # Tính điểm UCB1 cho mỗi agent
        ucb_scores = {}
        for agent_id in self.agent_ids:
            if self.agent_pulls[agent_id] == 0:
                ucb_scores[agent_id] = float('inf')
            else:
                avg_reward = self.agent_rewards[agent_id] / self.agent_pulls[agent_id]
                exploration_bonus = math.sqrt(2 * math.log(self.total_pulls) / self.agent_pulls[agent_id])
                ucb_scores[agent_id] = avg_reward + exploration_bonus
        
        # Chọn agent có điểm UCB1 cao nhất
                best_agent = max(ucb_scores, key=ucb_scores.get)
        print(f"  [MetaController] UCB scores: { {k: f'{v:.2f}' for k, v in ucb_scores.items()} } -> Chose {best_agent}")
        
        allocations = {agent_id: 0 for agent_id in self.agent_ids}
        allocations[best_agent] = slice_budget
        return allocations

    def update(self, rewards):
        for agent_id, reward in rewards.items():
            if reward > 0: # Chỉ cập nhật nếu agent đã chạy
                self.agent_rewards[agent_id] += reward
                self.agent_pulls[agent_id] += 1
                self.total_pulls += 1

In [14]:
# ----------------------------------------------------------------------------
# PHẦN 3: BỘ ĐIỀU PHỐI CHÍNH CỦA AMSCO
# ----------------------------------------------------------------------------

class AMSCO_Orchestrator:
    def __init__(self, total_budget, slice_budget):
        self.total_budget = total_budget
        self.slice_budget = slice_budget
        
        self.knowledge_hub = KnowledgeHub()
        
        self.agents = {
            "Random": RandomAgent("Random", SEARCH_SPACE, self.knowledge_hub),
            "Bayesian": BayesianAgent("Bayesian", SEARCH_SPACE, self.knowledge_hub),
            "Grid": GridAgent("Grid", SEARCH_SPACE, self.knowledge_hub)
        }
        agent_ids = list(self.agents.keys())
        
        self.performance_monitor = PerformanceMonitor(agent_ids)
        self.meta_controller = MetaController_UCB1(agent_ids)

    def run(self):
        """
        Thực thi vòng lặp tối ưu hóa chính của AMSCO.
        """
        current_budget = self.total_budget
        slice_num = 1
        
        while current_budget > 0:
            print(f"\n--- Slice {slice_num} | Budget remaining: {current_budget} ---")
            
            budget_for_slice = min(self.slice_budget, current_budget)
            
            # 1. Meta-Controller phân bổ ngân sách
            allocations = self.meta_controller.allocate(budget_for_slice)
            
            # 2. Chạy các agent (tuần tự trong demo này, có thể song song hóa)
            for agent_id, budget in allocations.items():
                if budget > 0:
                    self.agents[agent_id].run(budget)
            
            # 3. Cập nhật Performance Monitor
            all_trials = self.knowledge_hub.get_all_trials()
            self.performance_monitor.update(all_trials)
            
            # 4. Tính toán reward và cập nhật Meta-Controller
            rewards = self.performance_monitor.get_agent_rewards()
            self.meta_controller.update(rewards)
            
            current_budget -= budget_for_slice
            slice_num += 1
            
        # 5. Lấy kết quả cuối cùng từ Knowledge Hub
        print("\n--- Optimization Finished ---")
        final_result = self.knowledge_hub.get_best_trial()
        return final_result

In [15]:
# ----------------------------------------------------------------------------
# PHẦN 4: THỰC THI
# ----------------------------------------------------------------------------

if __name__ == "__main__":
    TOTAL_BUDGET = 100  # Tổng số lần thử
    SLICE_BUDGET = 10   # Số lần thử trong mỗi lát
    
    orchestrator = AMSCO_Orchestrator(total_budget=TOTAL_BUDGET, slice_budget=SLICE_BUDGET)
    best_trial = orchestrator.run()
    
    print("\n======================================")
    print("         AMSCO Final Result         ")
    print("======================================")
    print(f"Best Score (Accuracy): {best_trial['score']:.4f}")
    print("Best Hyperparameters:")
    for param, value in best_trial['params'].items():
        if isinstance(value, float):
            print(f"  - {param}: {value:.4f}")
        else:
            print(f"  - {param}: {value}")
    print("======================================")


--- Slice 1 | Budget remaining: 100 ---
  [MetaController] Initializing Random
    -> Running RandomAgent with budget: 10
  [KnowledgeHub] New best score: 0.9631 from Random
  [KnowledgeHub] New best score: 0.9649 from Random
  [KnowledgeHub] New best score: 0.9649 from Random
  [KnowledgeHub] New best score: 0.9666 from Random

--- Slice 2 | Budget remaining: 90 ---
  [MetaController] UCB scores: {'Random': '2.22', 'Bayesian': '1.98', 'Grid': '1.98'} -> Chose Random
    -> Running RandomAgent with budget: 10
  [KnowledgeHub] New best score: 0.9666 from Random

--- Slice 2 | Budget remaining: 90 ---
  [MetaController] UCB scores: {'Random': '2.22', 'Bayesian': '1.98', 'Grid': '1.98'} -> Chose Random
    -> Running RandomAgent with budget: 10
  [KnowledgeHub] New best score: 0.9684 from Random

--- Slice 3 | Budget remaining: 80 ---
  [MetaController] UCB scores: {'Random': '2.00', 'Bayesian': '1.84', 'Grid': '1.84'} -> Chose Random
    -> Running RandomAgent with budget: 10
  [Knowled